In [1]:
import torch, open_clip
from PIL import Image
from open_clip.tokenizer import SimpleTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME, PRETRAINED = "ViT-B-32", "laion2b_s34b_b79k"
model, _, preprocess = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED)
model = model.to(DEVICE).eval()
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
bpe = SimpleTokenizer()

@torch.inference_mode()
def encoder_textes(textes):
    v = model.encode_text(tokenizer(textes).to(DEVICE)).float()
    return (v / v.norm(dim=-1, keepdim=True).clamp_min(1e-12)).cpu().tolist()

@torch.inference_mode()
def encoder_image(chemin):
    with Image.open(chemin) as im:
        t = preprocess(im.convert("RGB")).unsqueeze(0).to(DEVICE)
    v = model.encode_image(t).float()
    return (v / v.norm(dim=-1, keepdim=True).clamp_min(1e-12)).cpu().tolist()[0]

def decouper_texte(texte, taille=60, chevauchement=10):
    tokens = bpe.encode(texte)
    for d in range(0, len(tokens), taille - chevauchement):
        frag = bpe.decode(tokens[d:d + taille]).strip()
        if frag:
            yield frag
        if d + taille >= len(tokens):
            break

c:\Users\DELL\rag-multimodal-agentique\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\DELL\rag-multimodal-agentique\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DELL\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, se

In [2]:
v_texte = encoder_textes(["a diagram of the transformer architecture"])[0]
print("Dimension :", len(v_texte))

Dimension : 512
